<a href="https://colab.research.google.com/github/naylaakas/PembelajaranMesin-2026/blob/main/JS06/Praktikum3-HNSW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install hnswlib

In [3]:
import hnswlib
import numpy as np
import time
from sklearn.neighbors import NearestNeighbors

# ===========================
# 1. Buat data 2D acak
# ===========================
num_elements = 1000
dim = 2
data = np.random.random((num_elements, dim)).astype(np.float32)

# Query point
query = np.array([[0.5, 0.5]], dtype=np.float32)
k = 5  # cari 5 tetangga terdekat

# ===========================
# 2. Exact NN (Brute Force)
# ===========================
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(data)

start = time.time()
distances, indices = nn.kneighbors(query)
end = time.time()

print("=== Exact NN ===")
print("Indices:", indices)
print("Distances:", distances)
print("Waktu:", end - start, "detik")

# ===========================
# 3. HNSW
# ===========================
# Inisialisasi index HNSW
p = hnswlib.Index(space='l2', dim=dim)

# Ukuran maksimum elemen yang bisa ditampung
p.init_index(max_elements=num_elements, ef_construction=100, M=16)

# Tambahkan data
p.add_items(data)

# Set parameter pencarian
p.set_ef(50)   # tradeoff speed vs accuracy

start = time.time()
labels, distances = p.knn_query(query, k=k)
end = time.time()

print("\n=== HNSW ===")
print("Indices:", labels)
print("Distances:", distances)
print("Waktu:", end - start, "detik")


=== Exact NN ===
Indices: [[834 376 571 532 665]]
Distances: [[0.01684858 0.01775609 0.01999155 0.02048686 0.02731943]]
Waktu: 0.07607150077819824 detik

=== HNSW ===
Indices: [[834 376 571 532 665]]
Distances: [[0.00028387 0.00031528 0.00039966 0.00041971 0.00074635]]
Waktu: 0.0002753734588623047 detik


In [7]:
# Konfigurasi percobaan
data_sizes = [1000, 1000000]
dimensions = [2, 5]

metrics = [
    ("Euclidean", "l2", "euclidean"),
    ("Angular/Cosine", "cosine", "cosine")
]

k = 5
results = []

# Melakukan semua kombinasi percobaan
for metric_idx, (metric_name, hnsw_space, exact_metric) in enumerate(metrics):
    for num_elements in data_sizes:
        for dim in dimensions:

            print(
                f"\nMetrik: {metric_name} | "
                f"Jumlah data: {num_elements} | Dimensi: {dim}D"
            )

            # 1. Membuat dataset
            rng = np.random.default_rng(
                42 + num_elements + dim + metric_idx * 100
            )

            data = rng.random(
                (num_elements, dim)
            ).astype(np.float32)

            # Query point
            query = np.full(
                (1, dim), 0.5, dtype=np.float32
            )

            # 2. Exact NN (Brute Force)
            nn = NearestNeighbors(
                n_neighbors=k,
                algorithm="brute",
                metric=exact_metric
            )

            nn.fit(data)

            start = time.time()
            distances_exact, indices_exact = nn.kneighbors(query)
            end = time.time()

            time_exact = end - start

            # 3. HNSW
            p = hnswlib.Index(
                space=hnsw_space,
                dim=dim
            )

            p.init_index(
                max_elements=num_elements,
                ef_construction=100,
                M=16
            )

            p.add_items(data)
            p.set_ef(50)

            start = time.time()
            labels, distances_hnsw = p.knn_query(query, k=k)
            end = time.time()

            time_hnsw = end - start

            # 4. Menampilkan hasil setiap percobaan
            print("=== Exact NN ===")
            print("Indices:", indices_exact[0])
            print("Distances:", distances_exact[0])
            print("Waktu:", round(time_exact * 1000, 4), "ms")

            print("\n=== HNSW ===")
            print("Indices:", labels[0])
            print("Distances:", distances_hnsw[0])
            print("Waktu:", round(time_hnsw * 1000, 4), "ms")

            # 5. Menyimpan hasil ke tabel
            results.append([
                metric_name,
                num_elements,
                f"{dim}D",
                indices_exact[0].tolist(),
                labels[0].tolist(),
                round(time_exact * 1000, 4),
                round(time_hnsw * 1000, 4)
            ])

            # Membersihkan variabel percobaan sebelumnya
            del data, query, nn, p


Metrik: Euclidean | Jumlah data: 1000 | Dimensi: 2D
=== Exact NN ===
Indices: [756 696 459 283 382]
Distances: [0.00602709 0.01330369 0.02127332 0.02453863 0.02460828]
Waktu: 1.652 ms

=== HNSW ===
Indices: [756 696 459 283 382]
Distances: [3.6325848e-05 1.7698818e-04 4.5255423e-04 6.0214440e-04 6.0556742e-04]
Waktu: 0.0882 ms

Metrik: Euclidean | Jumlah data: 1000 | Dimensi: 5D
=== Exact NN ===
Indices: [856   6 710 606 500]
Distances: [0.23027423 0.23319893 0.24739541 0.27042976 0.29348248]
Waktu: 0.6776 ms

=== HNSW ===
Indices: [856   6 710 606 500]
Distances: [0.05302622 0.05438174 0.06120449 0.07313226 0.08613197]
Waktu: 0.1166 ms

Metrik: Euclidean | Jumlah data: 1000000 | Dimensi: 2D
=== Exact NN ===
Indices: [519738 112018  77488 165319 323034]
Distances: [0.00040566 0.00074944 0.00087818 0.00093943 0.00094161]
Waktu: 26.1607 ms

=== HNSW ===
Indices: [519738 112018  77488 165319 323034]
Distances: [1.6456262e-07 5.6166436e-07 7.7119552e-07 8.8252438e-07 8.8663541e-07]
Waktu: